# AM — Data Availability Audit  *(v0.4.0 schema)*

**Author:** Aidan Meyers · Melaram Lab · TAMU-CC  
**Database:** Neon Postgres · project `aged-salad-62359207` · schema `aq` (v0.4.0)  
**Last updated:** 2026-06-02  

A clean, end-to-end data audit answering one question for every (site × pollutant × year) combination in the South Texas AQ pipeline:

> *How much data do we actually have, and where are the gaps?*

### What this notebook produces

1. **Inventory snapshot** — 42 active sites × parameter reference table from `aq.site_registry` and `aq.parameter_reference`.
2. **Hourly completeness matrix** — for every active criteria-pollutant site (`aq.pollutant_hourly`) compute observed hours / expected hours per year, plus first/last date observed.
3. **VOC coverage matrix** — same audit pattern applied to `aq.vocs_1hr` and `aq.vocs_24hr`.
4. **24hr-only sampler** — `aq.pollutant_daily_24hr` (Palo Alto PM10) coverage.
5. **Completeness heatmaps** — one per pollutant group, sites × years.
6. **Gantt-style availability dashboard** — interactive Plotly timeline, one row per (site × pollutant_group), bars span first → last observed, colored by completeness.
7. **Gap report** — site × pollutant × year combos below a configurable completeness threshold.
8. **CSV + HTML export** — everything dropped into `notebooks/reports/`.

### How to run in Colab

1. Open: <https://colab.research.google.com/github/AidanJMeyers/south-texas-aq-pipeline/blob/main/notebooks/AM_Data_Availability_Audit.ipynb>
2. 🔑 Add the secret `AQ_POSTGRES_URL` (Notebook access ON).
3. **Runtime → Run all.** ≈ 3–5 min.

Outputs land in `notebooks/reports/availability/`.

## 1. Setup, connection, helpers

In [ ]:
# One-time install (skip if running locally with these already present)
!pip install -q "psycopg[binary]" sqlalchemy pandas plotly matplotlib seaborn nbconvert

In [ ]:
import os, sys, json, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)

# ----- Melaram Lab brand palette -----
BRAND_NAVY     = '#213c4e'
BRAND_ORANGE   = '#c2410c'
BRAND_LIGHT_BG = '#F5F7F9'
BRAND_GRAY     = '#9aa6ad'
BRAND_OK       = '#2e7d4f'   # >= 90% completeness
BRAND_WARN     = '#e0a528'   # 50-90%
BRAND_BAD      = '#c2410c'   # < 50%
BRAND_MISSING  = '#d9dde0'   # no data at all

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   BRAND_LIGHT_BG,
    'axes.edgecolor':   BRAND_NAVY,
    'axes.labelcolor':  BRAND_NAVY,
    'xtick.color':      BRAND_NAVY,
    'ytick.color':      BRAND_NAVY,
    'axes.titlecolor':  BRAND_NAVY,
    'font.family':      'DejaVu Sans',
})

# ----- Output directory -----
try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPORT_DIR = Path('reports') / 'availability' if IN_COLAB else Path(__file__).resolve().parent / 'reports' / 'availability' if '__file__' in globals() else Path.cwd() / 'reports' / 'availability'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / 'figs').mkdir(exist_ok=True)
print(f'OK report dir: {REPORT_DIR}')

In [ ]:
# Pull connection URL from Colab Secrets if available, else env var
URL = None
try:
    from google.colab import userdata
    URL = userdata.get('AQ_POSTGRES_URL')
except Exception:
    pass
URL = URL or os.environ.get('AQ_POSTGRES_URL')
assert URL, 'Set AQ_POSTGRES_URL via Colab Secrets (key icon) or env var.'

# Force psycopg v3 dialect (the project standard)
if URL.startswith('postgresql://') and '+psycopg' not in URL:
    URL = 'postgresql+psycopg://' + URL[len('postgresql://'):]

engine = create_engine(URL, pool_pre_ping=True)
with engine.connect() as conn:
    ver = conn.execute(text('SELECT version()')).scalar()
print('OK connected:', ver[:80])

In [ ]:
# Audit configuration
AUDIT_START_YEAR = 2015
AUDIT_END_YEAR   = 2025
COMPLETENESS_OK    = 0.90    # >= 90% = green
COMPLETENESS_WARN  = 0.50    # 50-90% = yellow, <50% = red
GAP_THRESHOLD      = 0.75    # below this gets flagged in the gap report

def expected_hours(year: int) -> int:
    """8760 / 8784 hours per calendar year."""
    return 8784 if pd.Timestamp(year=year, month=1, day=1).is_leap_year else 8760

def expected_days(year: int) -> int:
    return 366 if pd.Timestamp(year=year, month=1, day=1).is_leap_year else 365

def completeness_color(pct: float) -> str:
    if pd.isna(pct):                    return BRAND_MISSING
    if pct >= COMPLETENESS_OK   * 100:  return BRAND_OK
    if pct >= COMPLETENESS_WARN * 100:  return BRAND_WARN
    return BRAND_BAD

print(f'Audit window: {AUDIT_START_YEAR} → {AUDIT_END_YEAR}')
print(f'Completeness bands: >={COMPLETENESS_OK*100:.0f}% OK · '
      f'{COMPLETENESS_WARN*100:.0f}-{COMPLETENESS_OK*100:.0f}% WARN · '
      f'<{COMPLETENESS_WARN*100:.0f}% BAD')

## 2. Site & parameter inventory snapshot

In [ ]:
sites = pd.read_sql(text("""
    SELECT aqsid, site_name, county_name, network, data_status,
           pollutants, n_pollutants, first_date, last_date, n_records,
           lat, lon, notes
    FROM aq.site_registry
    ORDER BY data_status, county_name, site_name
"""), engine)

params = pd.read_sql(text("""
    SELECT parameter_code, parameter_name, pollutant_group,
           units_of_measure, is_hap
    FROM aq.parameter_reference
    ORDER BY pollutant_group, parameter_code
"""), engine)

print(f'Sites in registry: {len(sites)}')
print(f'  active:    {(sites.data_status=="active").sum()}')
print(f'  reference: {(sites.data_status=="reference").sum()}')
print(f'  excluded:  {(sites.data_status=="excluded").sum()}')
print(f'  disabled:  {(sites.data_status=="disabled").sum()}')
print(f'Parameter codes: {len(params)}')

sites.to_csv(REPORT_DIR / 'site_registry_snapshot.csv', index=False)
params.to_csv(REPORT_DIR / 'parameter_reference_snapshot.csv', index=False)
sites.head(8)

## 3. Build the master availability matrix

One row per `(aqsid, site_name, county_name, pollutant_group, year)` with:
- `n_hours_observed` (hours that actually have a measurement)
- `n_days_observed` (distinct calendar days touched)
- `expected_hours` (8760/8784) and `expected_days` (365/366)
- `completeness_pct` (hourly resolution)
- `first_observed`, `last_observed`
- `source_table` (which table the row came from — `pollutant_hourly`, `pollutant_daily_24hr`, `vocs_1hr`, or `vocs_24hr`)

In [ ]:
# --- 3a. Criteria pollutants (aq.pollutant_hourly) ---
sql_hourly = text(f"""
    SELECT aqsid::text                              AS aqsid,
           site_name,
           county_name,
           pollutant_group,
           year::int                                AS year,
           COUNT(*)                                 AS n_hours_observed,
           COUNT(DISTINCT date_local)               AS n_days_observed,
           MIN(date_local)::text                    AS first_observed,
           MAX(date_local)::text                    AS last_observed
    FROM aq.pollutant_hourly
    WHERE sample_measurement IS NOT NULL
      AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, site_name, county_name, pollutant_group, year
""")
av_hourly = pd.read_sql(sql_hourly, engine)
av_hourly['source_table'] = 'pollutant_hourly'
av_hourly['resolution']   = '1hr'
print(f'pollutant_hourly groups: {len(av_hourly):,}')
av_hourly.head()

In [ ]:
# --- 3b. 24hr-only sampler (Palo Alto PM10 → aq.pollutant_daily_24hr) ---
sql_24hr = text(f"""
    SELECT aqsid::text                              AS aqsid,
           site_name,
           county_name,
           pollutant_group,
           EXTRACT(YEAR FROM date_local::date)::int AS year,
           -- 24hr samplers measure once per day → treat each day as 24 hours of coverage
           COUNT(*) * 24                            AS n_hours_observed,
           COUNT(DISTINCT date_local)               AS n_days_observed,
           MIN(date_local)::text                    AS first_observed,
           MAX(date_local)::text                    AS last_observed
    FROM aq.pollutant_daily_24hr
    WHERE EXTRACT(YEAR FROM date_local::date) BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, site_name, county_name, pollutant_group, year
""")
av_24hr = pd.read_sql(sql_24hr, engine)
av_24hr['source_table'] = 'pollutant_daily_24hr'
av_24hr['resolution']   = '24hr'
print(f'pollutant_daily_24hr groups: {len(av_24hr):,}')
av_24hr.head() if len(av_24hr) else 'no rows'

In [ ]:
# --- 3c. VOCs 1hr (AutoGC, 5 sites) ---
# Roll up all VOC parameter_codes into a single pseudo-pollutant_group "VOCs_1hr"
# (use parameter_code-level breakdown in the gap-detail section later).
sql_vocs_1hr = text(f"""
    SELECT aqsid::text                                 AS aqsid,
           MAX(site_name)                              AS site_name,
           MAX(county_name)                            AS county_name,
           'VOCs_1hr'                                  AS pollutant_group,
           EXTRACT(YEAR FROM date_local::date)::int    AS year,
           -- one row per (aqsid, datetime, parameter_code) → distinct hours = distinct datetimes
           COUNT(DISTINCT (date_local || ' ' || time_local)) AS n_hours_observed,
           COUNT(DISTINCT date_local)                  AS n_days_observed,
           MIN(date_local)::text                       AS first_observed,
           MAX(date_local)::text                       AS last_observed
    FROM aq.vocs_1hr
    WHERE sample_measurement IS NOT NULL
      AND EXTRACT(YEAR FROM date_local::date) BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, year
""")
av_vocs_1hr = pd.read_sql(sql_vocs_1hr, engine)
av_vocs_1hr['source_table'] = 'vocs_1hr'
av_vocs_1hr['resolution']   = '1hr'
print(f'vocs_1hr groups: {len(av_vocs_1hr):,}')
av_vocs_1hr.head()

In [ ]:
# --- 3d. VOCs 24hr (AutoGC, 8 sites) ---
sql_vocs_24hr = text(f"""
    SELECT aqsid::text                                 AS aqsid,
           MAX(site_name)                              AS site_name,
           MAX(county_name)                            AS county_name,
           'VOCs_24hr'                                 AS pollutant_group,
           EXTRACT(YEAR FROM date_local::date)::int    AS year,
           -- 24hr samples: count distinct days × 24 to keep the completeness axis consistent
           COUNT(DISTINCT date_local) * 24             AS n_hours_observed,
           COUNT(DISTINCT date_local)                  AS n_days_observed,
           MIN(date_local)::text                       AS first_observed,
           MAX(date_local)::text                       AS last_observed
    FROM aq.vocs_24hr
    WHERE sample_measurement IS NOT NULL
      AND EXTRACT(YEAR FROM date_local::date) BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, year
""")
av_vocs_24hr = pd.read_sql(sql_vocs_24hr, engine)
av_vocs_24hr['source_table'] = 'vocs_24hr'
av_vocs_24hr['resolution']   = '24hr'
print(f'vocs_24hr groups: {len(av_vocs_24hr):,}')
av_vocs_24hr.head()

In [ ]:
# --- 3e. Concat all four sources and add derived completeness columns ---
av = pd.concat([av_hourly, av_24hr, av_vocs_1hr, av_vocs_24hr],
               ignore_index=True)
av['expected_hours']    = av['year'].apply(expected_hours)
av['expected_days']     = av['year'].apply(expected_days)
av['completeness_pct']  = (av['n_hours_observed'] / av['expected_hours'] * 100).round(2)
av['day_coverage_pct']  = (av['n_days_observed']  / av['expected_days']  * 100).round(2)
av['first_observed']    = pd.to_datetime(av['first_observed'])
av['last_observed']     = pd.to_datetime(av['last_observed'])
av['span_days']         = (av['last_observed'] - av['first_observed']).dt.days + 1
av = av.sort_values(['pollutant_group', 'county_name', 'site_name', 'year']).reset_index(drop=True)

print(f'Master availability matrix: {len(av):,} rows')
print(f'  pollutant groups: {sorted(av.pollutant_group.unique())}')
print(f'  sites covered:    {av.aqsid.nunique()}')
print(f'  years covered:    {sorted(av.year.unique())}')
av.to_csv(REPORT_DIR / 'availability_matrix.csv', index=False)
av.head(10)

## 4. Dense (site × pollutant × year) lattice — fill in zeros for what's *missing*

The previous step only emits a row when there's at least one observation. To show **gaps as gaps** in the dashboard, build a full Cartesian product of:
- every active site,
- every pollutant group the site is *expected* to measure (per `site_registry.pollutants`),
- every year in the audit window,

then left-join the observed matrix onto it. Missing rows → 0 hours / NaN completeness.

In [ ]:
# Build the expected (site, pollutant_group) catalogue from site_registry.pollutants
active = sites[sites.data_status == 'active'].copy()
active['pollutant_list'] = active['pollutants'].fillna('').str.split(';').apply(
    lambda lst: [p.strip() for p in lst if p.strip()])
site_poll = active[['aqsid','site_name','county_name','pollutant_list']].explode('pollutant_list')
site_poll = site_poll.rename(columns={'pollutant_list':'pollutant_group'}).dropna(subset=['pollutant_group'])

# In v0.4.0, VOCs are split into 1hr + 24hr. Reflect that in expected coverage:
# any site that says it measures 'VOCs' is expected to have either VOCs_1hr or VOCs_24hr.
voc_rows = site_poll[site_poll.pollutant_group == 'VOCs'].copy()
voc_rows_1hr  = voc_rows.assign(pollutant_group='VOCs_1hr')
voc_rows_24hr = voc_rows.assign(pollutant_group='VOCs_24hr')
site_poll = pd.concat([
    site_poll[site_poll.pollutant_group != 'VOCs'],
    voc_rows_1hr,
    voc_rows_24hr,
], ignore_index=True)

years = list(range(AUDIT_START_YEAR, AUDIT_END_YEAR + 1))
expected = (site_poll
    .merge(pd.DataFrame({'year': years}), how='cross')
    .assign(expected_hours=lambda d: d.year.apply(expected_hours),
            expected_days =lambda d: d.year.apply(expected_days)))

# Left-join observations onto the expected lattice
merged = expected.merge(
    av[['aqsid','pollutant_group','year','n_hours_observed','n_days_observed',
        'first_observed','last_observed','source_table','resolution']],
    on=['aqsid','pollutant_group','year'],
    how='left',
)
merged['n_hours_observed'] = merged['n_hours_observed'].fillna(0).astype(int)
merged['n_days_observed']  = merged['n_days_observed'].fillna(0).astype(int)
merged['completeness_pct'] = (merged['n_hours_observed'] / merged['expected_hours'] * 100).round(2)
merged['day_coverage_pct'] = (merged['n_days_observed']  / merged['expected_days']  * 100).round(2)
merged['status_band']      = pd.cut(
    merged['completeness_pct'],
    bins=[-0.01, 0.0, COMPLETENESS_WARN*100, COMPLETENESS_OK*100, 200],
    labels=['MISSING','BAD','WARN','OK']
)

merged.to_csv(REPORT_DIR / 'availability_lattice.csv', index=False)
print(f'Dense lattice: {len(merged):,} rows '
      f'({merged.aqsid.nunique()} sites × {merged.pollutant_group.nunique()} pollutants × {len(years)} years)')
print('\nStatus band totals:')
print(merged.status_band.value_counts().to_string())
merged.head(8)

## 5. Completeness heatmaps — one panel per pollutant

Each panel: rows = sites (active only), columns = years, cell color = % completeness.  
Gray cells = site does not measure that pollutant. White cells = expected to measure but **zero data**.

In [ ]:
def make_heatmap(pollutant_group: str, ax):
    sub = merged[merged.pollutant_group == pollutant_group]
    if sub.empty:
        ax.set_visible(False); return
    pivot = sub.pivot_table(
        index   = 'site_name',
        columns = 'year',
        values  = 'completeness_pct',
        aggfunc = 'first',
    ).reindex(columns=years)
    # Sort sites: highest mean completeness first
    pivot = pivot.assign(_mean=pivot.mean(axis=1)).sort_values('_mean', ascending=False).drop(columns='_mean')

    sns.heatmap(
        pivot, vmin=0, vmax=100,
        cmap=sns.color_palette('YlGn', as_cmap=True),
        cbar_kws={'label':'% hourly completeness'},
        linewidths=0.4, linecolor='white',
        annot=True, fmt='.0f', annot_kws={'fontsize':7},
        ax=ax,
    )
    ax.set_title(f'{pollutant_group}  —  {len(pivot)} sites', color=BRAND_NAVY, fontweight='bold', pad=8)
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', labelsize=8, rotation=0)
    ax.tick_params(axis='y', labelsize=7)

groups = [g for g in ['Ozone','NOx_Family','PM2.5','PM10','CO','SO2','VOCs_1hr','VOCs_24hr']
          if g in merged.pollutant_group.unique()]
nrows = (len(groups) + 1) // 2
fig, axes = plt.subplots(nrows, 2, figsize=(20, 5.5*nrows))
axes = np.array(axes).reshape(-1)
for ax in axes: ax.set_visible(False)
for ax, g in zip(axes, groups):
    ax.set_visible(True)
    make_heatmap(g, ax)
fig.suptitle('Hourly completeness by site × year   (v0.4.0 schema)',
             color=BRAND_NAVY, fontsize=15, fontweight='bold', y=1.005)
fig.tight_layout()
fig.savefig(REPORT_DIR / 'figs' / 'completeness_heatmaps.png', dpi=140, bbox_inches='tight')
plt.show()

## 6. Gantt-style availability dashboard *(interactive)*

One row per `(site × pollutant_group)`. Bars span **first → last observed date** within each year. Color encodes completeness band (green/yellow/red/gray). Hover for exact stats. This is the headline visual — drop it straight into a slide deck or share the standalone HTML.

Built on `plotly.express.timeline` because it scales to hundreds of rows cleanly, exports to standalone HTML, and is brand-color-themeable.

In [ ]:
# Build the Gantt frame: only rows where there is at least 1 hour of data
g = merged[merged.n_hours_observed > 0].copy()
# Use first/last observed as the actual span; clamp to within the year
g['year_start'] = pd.to_datetime(g['year'].astype(str) + '-01-01')
g['year_end']   = pd.to_datetime(g['year'].astype(str) + '-12-31')
g['start']     = g[['first_observed','year_start']].max(axis=1)
g['finish']    = g[['last_observed','year_end']].min(axis=1)
g['row_label'] = g['site_name'] + '  ·  ' + g['county_name']
g['band']      = g['status_band'].astype(str)
g['hover']     = (
    g['site_name'] + ' (' + g['aqsid'] + ')<br>'
    + g['county_name'] + ' County<br>'
    + g['pollutant_group'] + ' · ' + g['year'].astype(str) + '<br>'
    + g['n_hours_observed'].map('{:,}'.format) + ' / '
    + g['expected_hours'].map('{:,}'.format) + ' hours '
    + '(' + g['completeness_pct'].map('{:.1f}'.format) + '%)<br>'
    + g['n_days_observed'].astype(str) + ' / '
    + g['expected_days'].astype(str) + ' days'
)

BAND_COLORS = {'OK': BRAND_OK, 'WARN': BRAND_WARN, 'BAD': BRAND_BAD,
               'MISSING': BRAND_MISSING, 'nan': BRAND_MISSING}

fig = px.timeline(
    g,
    x_start='start', x_end='finish',
    y='row_label',
    color='band',
    color_discrete_map=BAND_COLORS,
    facet_row='pollutant_group',
    category_orders={
        'pollutant_group': groups,
        'band': ['OK','WARN','BAD','MISSING'],
    },
    hover_name='hover',
    title='South Texas AQ — Data Availability Gantt   (v0.4.0 schema)',
)
fig.update_yaxes(autorange='reversed')   # alphabetical top-down
fig.update_layout(
    height=180 * len(groups),
    plot_bgcolor=BRAND_LIGHT_BG,
    paper_bgcolor='white',
    font=dict(family='Inter, DejaVu Sans, sans-serif', color=BRAND_NAVY),
    title_font=dict(size=18, color=BRAND_NAVY),
    legend_title_text='Completeness band',
    margin=dict(l=160, r=20, t=70, b=40),
)
fig.for_each_annotation(lambda a: a.update(
    text=a.text.split('=')[-1],
    font=dict(color=BRAND_NAVY, size=12),
))
fig.update_xaxes(showgrid=True, gridcolor='white')
fig.update_yaxes(showgrid=False, tickfont=dict(size=9))

out_html = REPORT_DIR / 'figs' / 'availability_gantt.html'
fig.write_html(out_html, include_plotlyjs='cdn')
print(f'OK interactive Gantt → {out_html}')
fig.show()

### 6b. Compact single-panel Gantt — one row per site × pollutant, full timeline

Same data, one row per `(site, pollutant_group)`, bars span every year that has data. Better for *executive summary* slides where you want a single image, not a faceted dashboard.

In [ ]:
# Collapse adjacent years into single bars where coverage is contiguous
g2 = g.sort_values(['aqsid','pollutant_group','year']).copy()
g2['key'] = g2['aqsid'] + '||' + g2['pollutant_group']

def collapse_runs(df):
    df = df.sort_values('year').reset_index(drop=True)
    out = []
    cur = None
    for _, row in df.iterrows():
        if cur is None:
            cur = row.to_dict(); cur['hours_sum'] = row.n_hours_observed; cur['exp_sum'] = row.expected_hours
        elif row.year == cur['year'] + 1:
            cur['finish']    = row.finish
            cur['year']      = row.year
            cur['hours_sum'] += row.n_hours_observed
            cur['exp_sum']   += row.expected_hours
        else:
            out.append(cur); cur = row.to_dict(); cur['hours_sum'] = row.n_hours_observed; cur['exp_sum'] = row.expected_hours
    if cur is not None: out.append(cur)
    return pd.DataFrame(out)

collapsed = (g2.groupby('key', group_keys=False)
               .apply(collapse_runs).reset_index(drop=True))
collapsed['avg_pct']  = (collapsed['hours_sum'] / collapsed['exp_sum'] * 100).round(1)
collapsed['band']     = pd.cut(collapsed['avg_pct'],
    bins=[-0.01, COMPLETENESS_WARN*100, COMPLETENESS_OK*100, 200],
    labels=['BAD','WARN','OK']).astype(str)
collapsed['row_label'] = collapsed['site_name'] + '  ·  ' + collapsed['pollutant_group']
collapsed['hover']     = (
    collapsed['site_name'] + ' (' + collapsed['aqsid'] + ')<br>'
    + collapsed['pollutant_group'] + '<br>'
    + collapsed['start'].dt.strftime('%Y-%m-%d') + ' → '
    + collapsed['finish'].dt.strftime('%Y-%m-%d') + '<br>'
    + 'avg completeness ' + collapsed['avg_pct'].map('{:.1f}'.format) + '%'
)

fig2 = px.timeline(
    collapsed.sort_values(['pollutant_group','site_name']),
    x_start='start', x_end='finish',
    y='row_label',
    color='band',
    color_discrete_map={'OK':BRAND_OK,'WARN':BRAND_WARN,'BAD':BRAND_BAD},
    hover_name='hover',
    title='South Texas AQ — collapsed availability runs (one row per site × pollutant)',
)
fig2.update_yaxes(autorange='reversed', tickfont=dict(size=9))
fig2.update_layout(
    height=max(700, 18 * collapsed['row_label'].nunique()),
    plot_bgcolor=BRAND_LIGHT_BG, paper_bgcolor='white',
    font=dict(family='Inter, DejaVu Sans, sans-serif', color=BRAND_NAVY),
    margin=dict(l=260, r=20, t=70, b=40),
)
fig2.update_xaxes(showgrid=True, gridcolor='white')
out_html2 = REPORT_DIR / 'figs' / 'availability_gantt_collapsed.html'
fig2.write_html(out_html2, include_plotlyjs='cdn')
print(f'OK collapsed Gantt → {out_html2}')
fig2.show()

## 7. Gap report — where coverage is below threshold

Anything below `GAP_THRESHOLD` (default 75%) in a year where the site *should* be measuring that pollutant. Sorted worst-first.

In [ ]:
gaps = merged[merged.completeness_pct < GAP_THRESHOLD * 100].copy()
gaps = gaps.sort_values(['completeness_pct','pollutant_group','site_name'])
gaps = gaps[['aqsid','site_name','county_name','pollutant_group','year',
             'n_hours_observed','expected_hours','completeness_pct',
             'n_days_observed','expected_days','day_coverage_pct',
             'first_observed','last_observed','status_band','source_table']]
gaps.to_csv(REPORT_DIR / 'gap_report.csv', index=False)
print(f'Gap rows (< {GAP_THRESHOLD*100:.0f}% completeness): {len(gaps):,}')
print(f'  fully missing (0 hours): {(gaps.completeness_pct == 0).sum():,}')
print(f'  partial gap (>0, <{GAP_THRESHOLD*100:.0f}%): {((gaps.completeness_pct > 0) & (gaps.completeness_pct < GAP_THRESHOLD*100)).sum():,}')
print()
print('Worst 25 (highest priority to investigate):')
display(gaps.head(25))

In [ ]:
# Roll-ups for the headline numbers in the narrative section below
summary_by_pollutant = (merged
    .groupby('pollutant_group')
    .agg(n_site_years   = ('aqsid', 'size'),
         n_sites        = ('aqsid', 'nunique'),
         mean_completeness = ('completeness_pct', 'mean'),
         pct_ok         = ('status_band', lambda s: (s=='OK').mean() * 100),
         pct_missing    = ('status_band', lambda s: (s=='MISSING').mean() * 100))
    .round(2)
    .sort_values('mean_completeness', ascending=False))
summary_by_pollutant.to_csv(REPORT_DIR / 'summary_by_pollutant.csv')

summary_by_year = (merged
    .groupby('year')
    .agg(n_site_pollutants = ('aqsid','size'),
         mean_completeness = ('completeness_pct','mean'),
         pct_ok       = ('status_band', lambda s: (s=='OK').mean() * 100),
         pct_missing  = ('status_band', lambda s: (s=='MISSING').mean() * 100))
    .round(2))
summary_by_year.to_csv(REPORT_DIR / 'summary_by_year.csv')

print('=== Summary by pollutant group ===')
print(summary_by_pollutant.to_string())
print()
print('=== Summary by year ===')
print(summary_by_year.to_string())

## 8. Narrative summary

*Markdown cell below is auto-rendered in the exported HTML report.*

In [ ]:
from IPython.display import Markdown, display
n_sites_active   = (sites.data_status == 'active').sum()
n_lattice        = len(merged)
n_with_data      = (merged.n_hours_observed > 0).sum()
n_full           = (merged.status_band == 'OK').sum()
n_missing        = (merged.status_band == 'MISSING').sum()
n_partial_warn   = (merged.status_band == 'WARN').sum()
n_partial_bad    = (merged.status_band == 'BAD').sum()
best_pollutant   = summary_by_pollutant.index[0]
best_pct         = summary_by_pollutant.iloc[0]['mean_completeness']
worst_pollutant  = summary_by_pollutant.index[-1]
worst_pct        = summary_by_pollutant.iloc[-1]['mean_completeness']

md = f'''
### Headline numbers

- **{n_sites_active} active sites** × **{len(groups)} pollutant groups** × **{len(years)} years** = **{n_lattice:,}** site-pollutant-year cells in the expected coverage lattice.
- **{n_with_data:,} ({n_with_data/n_lattice*100:.1f}%)** cells have at least some data.
- **{n_full:,} ({n_full/n_lattice*100:.1f}%)** meet the OK band (≥ {COMPLETENESS_OK*100:.0f}% hourly completeness).
- **{n_partial_warn:,}** in WARN band, **{n_partial_bad:,}** in BAD band, **{n_missing:,}** fully MISSING.
- Best-covered pollutant: **{best_pollutant}** at **{best_pct:.1f}%** mean completeness.
- Worst-covered pollutant: **{worst_pollutant}** at **{worst_pct:.1f}%** mean completeness.

### How to read the Gantt

Each bar is one site-year. Green = nearly complete, yellow = partial, red = sparse, gray = nothing in that cell. Hover for exact hour counts. The dashboard makes three patterns immediately visible:

1. **Lifecycle gaps** — sites that came online mid-period (e.g. 2018 onward) show empty left side.
2. **Episodic instrument issues** — a single red year sandwiched between green ones.
3. **Sensor retirements** — solid green that abruptly stops, never resumes.

### What to do with the gap report

- **`MISSING` rows where the site genuinely never measured that pollutant** → trim `site_registry.pollutants` so the lattice stops expecting it.
- **`MISSING` rows where TCEQ TAMIS *should* have data** → candidate for the next refresh batch (see `EPA_Refresh_2025_AM.py` / `TCEQ_Append_2025_AM.py`).
- **`BAD`/`WARN` rows with > 0 data** → instrument calibration or downtime episode; cross-reference with TCEQ's data validation reports for that quarter.
'''
display(Markdown(md))

## 9. Export everything to a single shareable HTML report

Generates `notebooks/reports/availability/AM_Data_Availability_Audit.html` — a standalone file with **all code blocks visible** plus the rendered tables, heatmaps, and a link to the interactive Gantt HTML. Share that single file with the lab.

In [ ]:
import subprocess
notebook_path = Path.cwd() / 'AM_Data_Availability_Audit.ipynb'
if not notebook_path.exists():
    # In Colab, find the notebook by its mounted name
    candidates = list(Path('/content').rglob('AM_Data_Availability_Audit.ipynb'))
    if candidates:
        notebook_path = candidates[0]

html_out = REPORT_DIR / 'AM_Data_Availability_Audit.html'
if notebook_path.exists():
    cmd = [
        sys.executable, '-m', 'nbconvert',
        '--to', 'html',
        '--output', str(html_out),
        '--no-input' if False else '--TemplateExporter.exclude_input=False',
        str(notebook_path),
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        print(f'OK HTML report → {html_out}')
    else:
        print('nbconvert stderr:', res.stderr[:500])
else:
    print('Notebook path not found for nbconvert; run the cell above with a known path.')

print()
print('=== All outputs ===')
for p in sorted(REPORT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(REPORT_DIR)}  ({p.stat().st_size/1024:.1f} kB)')

---

**End of audit.** Drop the HTML in Teams / SharePoint, or post the Gantt link directly. For any flagged gap, the `gap_report.csv` row gives you the exact `(aqsid, pollutant_group, year)` to chase in TCEQ TAMIS.